In [15]:
import json
import os

with open("../result/output.json", "r") as f:
    test = json.load(f)

test

FileNotFoundError: [Errno 2] No such file or directory: '../result/output.json'

In [6]:
%load_ext tensorboard
%tensorboard --logdir "result2/train_cord/test_experiment/" --port=8009


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [2]:
import torch
from transformers import (
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    DonutProcessor
)
from PIL import Image
import re
import os
import sys


def perform_donut_inference(model, processor, image_path, task_prompt, device):
    """
    Provede inferenci s načteným Donut modelem.
    """
    if not os.path.exists(image_path):
        print(f"Chyba: Vstupní obrázek nebyl nalezen: {image_path}")
        return None

    try:
        print(f"Načítám obrázek: {image_path}")
        image = Image.open(image_path).convert("RGB")

        print("Zpracovávám vstup pomocí processoru...")
        pixel_values = processor(image, return_tensors="pt").pixel_values
        decoder_input_ids = processor.tokenizer(
            task_prompt, add_special_tokens=False, return_tensors="pt"
        ).input_ids

        pixel_values = pixel_values.to(device)
        decoder_input_ids = decoder_input_ids.to(device)

        print("Provádím generování (inferenci)...")
        with torch.no_grad(): # Ujistíme se, že se nepočítají gradienty
             outputs = model.generate(
                 pixel_values,
                 decoder_input_ids=decoder_input_ids,
                 max_length=model.decoder.config.max_position_embeddings,
                 pad_token_id=processor.tokenizer.pad_token_id,
                 eos_token_id=processor.tokenizer.eos_token_id,
                 use_cache=True,
                 bad_words_ids=[[processor.tokenizer.unk_token_id]],
                 return_dict_in_generate=True,
             )

        print("Dekóduji výstup...")
        sequence = processor.batch_decode(outputs.sequences)[0]
        sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
        sequence = re.sub(r"^{}".format(re.escape(task_prompt)), "", sequence).strip()

        # Volitelně zde můžete přidat parsování, např. processor.token2json(sequence)
        # if task_prompt == "<s_cord-v2>":
        #     try:
        #         return processor.token2json(sequence)
        #     except Exception:
        #         print("Nepodařilo se parsovat jako JSON, vracím text.")
        #         return sequence

        return sequence # Vrátí vyčištěný text

    except Exception as e:
        print(f"Chyba během inference: {e}")
        import traceback
        traceback.print_exc()
        return None

In [9]:
import torch
from transformers import (
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    DonutProcessor
)
from safetensors.torch import load_file # Ujistěte se, že máte nainstalováno: pip install safetensors
import os
import sys
import traceback

def load_donut_model_from_checkpoint(
    base_model_id: str, # Např. "naver-clova-ix/donut-base" - Zkontrolujte, z čeho jste trénovali!
    checkpoint_path: str, # Cesta k vašemu model.safetensors
    device: torch.device
):
    if not os.path.exists(checkpoint_path):
        print(f"Chyba: Checkpoint soubor nebyl nalezen na cestě: {checkpoint_path}")
        return None, None

    try:
        print(f"Načítám processor a config z '{base_model_id}'...")
        # Načítáme processor a config z původního base modelu, NE z vašeho adresáře,
        # protože váš config.json má špatný formát pro VisionEncoderDecoderModel.
        try:
             processor = DonutProcessor.from_pretrained(base_model_id)
             config = VisionEncoderDecoderConfig.from_pretrained(base_model_id)
        except Exception as e:
             print(f"CHYBA: Nepodařilo se načíst processor/config z '{base_model_id}'. Zkontrolujte název! Chyba: {e}")
             return None, None

        print("Vytvářím instanci modelu z konfigurace stažené z Hugging Face...")
        # Vytvoříme model se správnou architekturou podle base_model_id
        model = VisionEncoderDecoderModel(config=config)

        print(f"Načítám checkpoint z '{checkpoint_path}'...")

        state_dict = None
        # --- Rozlišení mezi .ckpt a .safetensors ---
        if checkpoint_path.endswith(".safetensors"):
            print("Detekován soubor .safetensors, načítám pomocí safetensors.torch.load_file...")
            # Načteme na CPU pro manipulaci s klíči
            raw_state_dict = load_file(checkpoint_path, device='cpu')
            print("Provádím úpravu klíčů ze .safetensors kvůli neshodě prefixů...")

            state_dict = {}
            # Toto jsou prefixy z VAŠEHO předchozího výpisu Unexpected keys
            prefix_encoder_in_file = "encoder.model."
            prefix_decoder_in_file = "decoder.model.model."
            # Toto jsou prefixy, které model OČEKÁVÁ (z Missing keys)
            prefix_encoder_expected = "encoder."
            prefix_decoder_expected = "decoder.model." # Jen jeden 'model'
            prefix_lm_head_in_file = "decoder.model.lm_head." # Možná také potřebuje upravit
            prefix_lm_head_expected = "decoder.lm_head."

            len_prefix_encoder = len(prefix_encoder_in_file)
            len_prefix_decoder = len(prefix_decoder_in_file)
            len_prefix_lm_head = len(prefix_lm_head_in_file)

            fixed_keys_count = 0
            kept_keys_count = 0
            for key, value in raw_state_dict.items():
                new_key = key
                if key.startswith(prefix_encoder_in_file):
                    # Nahradit 'encoder.model.' za 'encoder.'
                    new_key = prefix_encoder_expected + key[len_prefix_encoder:]
                    fixed_keys_count += 1
                elif key.startswith(prefix_decoder_in_file):
                    # Nahradit 'decoder.model.model.' za 'decoder.model.'
                    new_key = prefix_decoder_expected + key[len_prefix_decoder:]
                    fixed_keys_count += 1
                elif key.startswith(prefix_lm_head_in_file):
                     # Nahradit 'decoder.model.lm_head.' za 'decoder.lm_head.'
                     new_key = prefix_lm_head_expected + key[len_prefix_lm_head:]
                     fixed_keys_count +=1
                else:
                    # Klíč nemá žádný z problematických prefixů, ponecháme ho
                    kept_keys_count += 1
                    # Můžete odkomentovat pro ladění, pokud stále budou problémy:
                    # print(f"  Ponechávám klíč beze změny: {key}")

                state_dict[new_key] = value
            print(f"Dokončena úprava klíčů: {fixed_keys_count} klíčů upraveno, {kept_keys_count} klíčů ponecháno.")

        elif checkpoint_path.endswith(".ckpt"):
            # ... Vaše logika pro .ckpt ...
            # !! DŮLEŽITÉ: Pokud byste načítali .ckpt, i zde byste museli
            #    po extrakci `checkpoint['state_dict']` aplikovat podobnou logiku
            #    pro kontrolu a opravu prefixů, pokud by byly nesprávné.
             print("Načítání .ckpt - zkontrolujte logiku pro prefixy!")
             # Zde by měla být vaše logika pro načtení state_dict z .ckpt
             # a následně kód pro odstranění/opravu prefixů (např. 'model.')
             # Pokud váš .ckpt neobsahuje state_dict, tato cesta selže.
             checkpoint = torch.load(checkpoint_path, map_location='cpu')
             if 'state_dict' in checkpoint:
                 state_dict = checkpoint['state_dict']
                 # Zde aplikovat čištění prefixů pro state_dict z .ckpt
                 # ... (např. odstranění 'model.') ...
             else:
                 print(f"CHYBA: V .ckpt souboru '{checkpoint_path}' chybí klíč 'state_dict'.")
                 return None, None

        else:
            print(f"CHYBA: Nepodporovaný typ souboru checkpointu: {checkpoint_path}")
            return None, None
        # ------ Konec rozlišení ------

        if state_dict is None:
             print("CHYBA: State dict nebyl úspěšně načten nebo zpracován.")
             return None, None

        print("Načítám upravené váhy do modelu...")
        try:
            # Nyní, když jsme opravili klíče, zkusíme strict=True
            missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=True)
            # Pokud by i po opravě něco chybělo/přebývalo, vypíše se to:
            if missing_keys:
                print("\nCHYBA i po úpravě: Některé klíče stále chyběly v state_dict:")
                print(missing_keys)
                # return None, None # Možná zde ukončit?
            if unexpected_keys:
                print("\nVarování i po úpravě: Některé klíče ze state_dict nebyly v modelu:")
                print(unexpected_keys)

        except RuntimeError as e:
            print(f"\nCHYBA při load_state_dict (i po úpravě klíčů): {e}")
            print("   Zkontrolujte znovu `base_model_id` a logiku úpravy prefixů.")
            print("   Můžete zkusit znovu se strict=False, ale výsledek bude nejistý.")
            # return None, None # Ukončit zde?

        model.eval()
        model.to(device) # Přesun na cílové zařízení až po načtení vah
        print(f"Model úspěšně načten a přesunut na '{device}'.")
        return model, processor

    except Exception as e:
        print(f"Nastala neočekávaná chyba při načítání: {e}")
        traceback.print_exc()
        return None, None

# ---- !!! ZDE NASTAVTE SPRÁVNÉ HODNOTY !!! ----

# Z jakého modelu jste PŮVODNĚ začali finetuning?
# Pravděpodobně 'naver-clova-ix/donut-base' pokud jste začínali od nuly.
# Pokud jste pokračovali v tréninku již finetunovaného, ponechte ten.
BASE_MODEL_ID = "naver-clova-ix/donut-base-finetuned-cord-v2" # Nebo "naver-clova-ix/donut-base-finetuned-cord-v2"

# Cesta k vašemu .safetensors souboru
CHECKPOINT_PATH = "result2/train_cord/test_experiment/model.safetensors"  # Použijte tento!

# Cesta k obrázku
# IMAGE_PATH = "../dataset_creating_json/dataset/validation/785f9069-d049-4da6-b208-217b0c5718d3.d3c3a289-e5ac-11e9-9fda-00155d012102.21.None.svkhk.jpg" # Nahraďte skutečnou cestou
IMAGE_PATH = "../dataset_creating_json/dataset/train/0bd784f0-01b7-11ed-bd12-005056827e51.107.c59671c0-3145-464f-a0c5-726c354ea478.jpg"  # Nahraďte skutečnou cestou
IMAGE_PATH = "../dataset_creating_json/dataset/train/0ac090d0-2d72-11e2-89c9-005056827e51.412e7e80-b0ed-11e2-8c63-5ef3fc9ae867.161.None.mzk.jpg"  # Nahraďte skutečnou cestou
# IMAGE_PATH = "../dataset_creating_json/dataset/validation/00acff12-f2e0-4b32-a568-619c58cabf70.69a89489-5d0a-11e5-a52d-001b21d0d3a4.4.None.kvkli.jpg" # Nahraďte skutečnou cestou

# Task prompt (ujistěte se, že je správný!)
TASK_PROMPT = "<s_dataset>" # Např. "<s_cord-v2>" nebo "<s_cord-v2-ocr>"

# -------------------------------------------------

# Kontrola obrázku a zařízení... (váš kód)
if not os.path.exists(IMAGE_PATH):
    print(f"CHYBA: Zadaný obrázek '{IMAGE_PATH}' neexistuje.")
    sys.exit(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám zařízení: {device}")

# 1. Načtení modelu pomocí upravené funkce
model, processor = load_donut_model_from_checkpoint(BASE_MODEL_ID, CHECKPOINT_PATH, device)

# 2. Inference (váš kód)
if model and processor:
    # Zde přidejte definici vaší funkce perform_donut_inference, pokud není globální
    # def perform_donut_inference(model, processor, image_path, task_prompt, device): ...

    result = perform_donut_inference(model, processor, IMAGE_PATH, TASK_PROMPT, device)

    if result is not None:
        print("\n--- Výsledek Inference ---")
        if isinstance(result, dict):
            import json
            print(json.dumps(result, indent=2, ensure_ascii=False))
        else:
            print(result)
        print("--------------------------")
    else:
        print("\nInference se nezdařila.")
else:
    print("\nNepodařilo se načíst model, inference se neprovede.")


# ---- Nezapomeňte definovat funkci perform_donut_inference ----
def perform_donut_inference(model, processor, image_path, task_prompt, device):
    """
    Provede inferenci s načteným Donut modelem.
    """
    if not os.path.exists(image_path):
        print(f"Chyba: Vstupní obrázek nebyl nalezen: {image_path}")
        return None

    try:
        print(f"Načítám obrázek: {image_path}")
        image = Image.open(image_path).convert("RGB")

        print("Zpracovávám vstup pomocí processoru...")
        # Zajistíme použití správného processoru načteného funkcí
        pixel_values = processor(image, return_tensors="pt").pixel_values
        decoder_input_ids = processor.tokenizer(
            task_prompt, add_special_tokens=False, return_tensors="pt"
        ).input_ids

        pixel_values = pixel_values.to(device)
        decoder_input_ids = decoder_input_ids.to(device)

        print("Provádím generování (inferenci)...")
        with torch.no_grad():
             outputs = model.generate(
                 pixel_values,
                 decoder_input_ids=decoder_input_ids,
                 max_length=model.decoder.config.max_position_embeddings,
                 pad_token_id=processor.tokenizer.pad_token_id,
                 eos_token_id=processor.tokenizer.eos_token_id,
                 use_cache=True,
                 bad_words_ids=[[processor.tokenizer.unk_token_id]],
                 return_dict_in_generate=True,
             )

        print("Dekóduji výstup...")
        sequence = processor.batch_decode(outputs.sequences)[0]
        sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
        sequence = re.sub(r"^{}".format(re.escape(task_prompt)), "", sequence).strip()

        # Volitelné parsování
        # if task_prompt == "<s_cord-v2>":
        #    ...

        return sequence

    except Exception as e:
        print(f"Chyba během inference: {e}")
        traceback.print_exc()
        return None
# -------------------------------------------------------------

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.


Používám zařízení: cpu
Načítám processor a config z 'naver-clova-ix/donut-base-finetuned-cord-v2'...
Vytvářím instanci modelu z konfigurace stažené z Hugging Face...
Načítám checkpoint z './result2/train_cord/test_experiment/model.safetensors'...
Detekován soubor .safetensors, načítám pomocí safetensors.torch.load_file...
Provádím úpravu klíčů ze .safetensors kvůli neshodě prefixů...
Dokončena úprava klíčů: 413 klíčů upraveno, 0 klíčů ponecháno.
Načítám upravené váhy do modelu...

CHYBA při load_state_dict (i po úpravě klíčů): Error(s) in loading state_dict for VisionEncoderDecoderModel:
	Missing key(s) in state_dict: "encoder.embeddings.patch_embeddings.projection.weight", "encoder.embeddings.patch_embeddings.projection.bias", "encoder.embeddings.norm.weight", "encoder.embeddings.norm.bias", "encoder.encoder.layers.0.blocks.0.layernorm_before.weight", "encoder.encoder.layers.0.blocks.0.layernorm_before.bias", "encoder.encoder.layers.0.blocks.0.attention.self.relative_position_bias_tab

In [6]:
# ... (váš kód pro načtení obrázku, zařízení, atd.) ...

# 1. Načtení modelu pomocí upravené funkce
print("Zahajuji načítání modelu pro následné uložení...")

# 2. Ověření a ULOŽENÍ do nového formátu
if model and processor:
    print("\nModel a processor úspěšně načteny.")
    print("Nyní ukládám do standardního formátu Hugging Face...")

    # ---- Zvolte název nového adresáře ----
    new_save_directory = "./muj_donut_hf_format"
    # ------------------------------------

    try:
        # Vytvoříme adresář, pokud neexistuje
        os.makedirs(new_save_directory, exist_ok=True)

        # Uložíme model (váhy + správný config.json)
        model.save_pretrained(new_save_directory)
        print(f"Model uložen do: {new_save_directory}")

        # Uložíme processor (všechny jeho soubory)
        processor.save_pretrained(new_save_directory)
        print(f"Processor uložen do: {new_save_directory}")

        print("\nÚspěšně uloženo! Nyní můžete tento adresář načíst pomocí:")
        print(f"model = VisionEncoderDecoderModel.from_pretrained('{new_save_directory}')")
        print(f"processor = DonutProcessor.from_pretrained('{new_save_directory}')")

    except Exception as e:
        print(f"\nChyba při ukládání modelu/processoru: {e}")
        import traceback
        traceback.print_exc()

    # Tady může následovat vaše původní inferenční logika, pokud ji chcete hned otestovat
    # print("\nSpouštím inferenci pro test...")
    # result = perform_donut_inference(model, processor, IMAGE_PATH, TASK_PROMPT, device)
    # ... (výpis výsledku) ...

else:
    print("\nNepodařilo se načíst model, nelze uložit ani provést inferenci.")

# --- Nezapomeňte na definice funkcí load_donut_model_from_checkpoint a perform_donut_inference ---
# ... (definice těchto funkcí) ...

Zahajuji načítání modelu pro následné uložení...

Model a processor úspěšně načteny.
Nyní ukládám do standardního formátu Hugging Face...
Model uložen do: ./muj_donut_hf_format
Processor uložen do: ./muj_donut_hf_format

Úspěšně uloženo! Nyní můžete tento adresář načíst pomocí:
model = VisionEncoderDecoderModel.from_pretrained('./muj_donut_hf_format')
processor = DonutProcessor.from_pretrained('./muj_donut_hf_format')


In [17]:
from model import DonutModel
model = DonutModel.from_pretrained("../../donut_training/result2/train_cord/test_experiment/")

ModuleNotFoundError: No module named 'donut.model'; 'donut' is not a package

In [1]:
print("Načítám obrázek...")

Načítám obrázek...
